# ECMWF TC Tracks & Wind Pipeline

This notebook demonstrates the TC track and wind processing pipeline: from BUFR download through to ensemble wind threshold envelopes.

For precipitation, see `pipeline_demonstration_precipitation.ipynb`.

## Overview

| Step | Module | Output |
|---|---|---|
| 1. Download TC Data | `ecmwf_tc_data_downloader` | Single BUFR4 per run (all storms, all members) |
| 2. Extract BUFR | `ecmwf_tc_data_extractor` | Per-storm CSVs (one file per named storm) |
| 3. Transform | `ecmwf_tc_data_transformer` | Standardised tracks with wind radii + WKT polygons |
| 4. Download Wind | `ecmwf_wind_data_downloader` | Ensemble 10m wind GRIB files (`type="pf"/"cf"`) |
| 5. Wind Combination | `ecmwf_tc_wind_combination` | Wind threshold contours, unioned across forecast steps |

| Output file | Snowflake table | Contents |
|---|---|---|
| `*_transformed.csv` | `TC_TRACKS` | One row per member × step, with wind radii and polygons |
| `*_envelopes_individual.csv` | `TC_ENVELOPES_INDIVIDUAL` | Wind threshold polygon per member × step |
| `*_envelopes_combined.csv` | `TC_ENVELOPES_COMBINED` | Unioned wind polygon per member × threshold |

## Setup


In [1]:
import os
import pandas as pd
from pathlib import Path

# Pipeline modules
from ecmwf_tc_data_downloader import download_tc_data, list_downloaded_files
from ecmwf_tc_data_extractor import extract_tc_data, filter_tc_data, save_per_storm_csvs
from ecmwf_tc_data_transformer import transform_tc_data_from_file
from ecmwf_wind_data_downloader import download_ensemble_wind
from ecmwf_tc_wind_combination import process_wind_combination

In [2]:
# ── Date / run-time selection ──────────────────────────────────────────────
#
# ECMWF publishes four TC track forecast runs per day:
#   00Z  (published ~09:00 UTC)  →  240-hour track horizon
#   06Z  (published ~13:00 UTC)  →  144-hour track horizon
#   12Z  (published ~21:00 UTC)  →  240-hour track horizon
#   18Z  (published ~01:00 UTC)  →  144-hour track horizon
#
# Leave SELECTED_DATE empty ("") to download the most recent available run.

SELECTED_DATE    = ""           # "YYYYMMDD" or "" for latest
SELECTED_RUNTIME = "00"         # "00" / "06" / "12" / "18", or "" for latest

if SELECTED_DATE:
    print(f"Target: {SELECTED_DATE} {SELECTED_RUNTIME or 'any'}Z")
else:
    print("Target: latest available forecast (determined by ecmwf-opendata client)")

Target: latest available forecast (determined by ecmwf-opendata client)


In [3]:
# Example override — uncomment and set to target a specific run:
# SELECTED_DATE    = "20251109"
# SELECTED_RUNTIME = "00"

In [4]:
# Create output directories
directories = {
    'tc_data':        'tc_data',          # downloaded BUFR files + extracted per-storm CSVs
    'tc_transformed': 'tc_transformed',   # transformed TC CSVs
    'wind_data':      'wind_data',        # wind GRIB files
    'wind_extracted': 'wind_extracted',   # wind envelope CSVs
}

for dir_path in directories.values():
    Path(dir_path).mkdir(exist_ok=True)

## Step 1: Download TC Data


In [5]:
# Download TC data — returns a single combined BUFR4 file per forecast run
# (all active storms × all 51 ensemble members in one file)
result = download_tc_data(
    date=SELECTED_DATE,
    run_time=SELECTED_RUNTIME,
    output_dir=directories['tc_data'],
)
print(f"Downloaded: {result['downloaded']}  Failed: {result['failed']}")

bufr_files = list(Path(directories['tc_data']).glob("*.bufr4"))
print(f"BUFR files: {[f.name for f in bufr_files]}")

2026-07-01 17:01:04,704 - WARNING - To ensure the stability of our systems and to preserve resources for our operational activities (network, compute, etc.), access to the open-data portal is limited to 500 simultaneous connections. This limit helps us guarantee reliable service for our operational users, especially during periods of high demand. For added reliability, the open-data is replicated across AWS, Azure, and Google Cloud. If you experience difficulties accessing the portal directly, you can also retrieve the data from these cloud platforms.
2026-07-01 17:01:04,705 - INFO - Downloading TC tracks: 2026-07-01 00Z (step=240h)
2026-07-01 17:01:04,706 - INFO - Downloading https://data.ecmwf.int/forecasts/20260701/00z/ifs/0p25/enfo/20260701000000-240h-enfo-tf.bufr
2026-07-01 17:01:05,249 - WARNING - data.ecmwf.int failed for tc_tracks_2026-07-01_r00.bufr4: 404 Client Error: Not Found for url: https://data.ecmwf.int/forecasts/20260701/00z/ifs/0p25/enfo/20260701000000-240h-enfo-tf.bu

Downloaded: 1  Failed: 0
BUFR files: ['tc_tracks_2026-06-30_r18.bufr4', 'tc_tracks_2026-07-01_r00.bufr4']


## Step 2: Extract BUFR Data


In [6]:
# Extract all storms from the combined BUFR file,
# filter to named storms, and write one CSV per storm.
per_storm_csvs = []

for bufr_file in bufr_files:
    print(f"Extracting: {bufr_file.name}")
    df = extract_tc_data(str(bufr_file), verbose=False)

    if df.empty:
        print("  No data extracted")
        continue

    df = filter_tc_data(df, named_storms_only=True)
    if df.empty:
        print("  No named storms found")
        continue

    storms = df['storm_id'].unique()
    print(f"  Named storms: {list(storms)}")

    csv_paths = save_per_storm_csvs(df, directories['tc_data'], bufr_file.name, verbose=True)
    per_storm_csvs.extend(csv_paths)

print(f"\nPer-storm CSVs written: {len(per_storm_csvs)}")
for p in per_storm_csvs:
    print(f"  {p}")

Extracting: tc_tracks_2026-06-30_r18.bufr4
Number of Ensemble Members:  51
Number of Ensemble Members:  48
Number of Ensemble Members:  31
Number of Ensemble Members:  12
Number of Ensemble Members:  4
Number of Ensemble Members:  51
Number of Ensemble Members:  36
Number of Ensemble Members:  24
Number of Ensemble Members:  10
Number of Ensemble Members:  3
Number of Ensemble Members:  1
Number of Ensemble Members:  51
Number of Ensemble Members:  51
Number of Ensemble Members:  26
Number of Ensemble Members:  7
Number of Ensemble Members:  2
Number of Ensemble Members:  18
Number of Ensemble Members:  2
Number of Ensemble Members:  9
Number of Ensemble Members:  9
Number of Ensemble Members:  2
Number of Ensemble Members:  36
Number of Ensemble Members:  10
Number of Ensemble Members:  3
Number of Ensemble Members:  10
Number of Ensemble Members:  3
Number of Ensemble Members:  32
Number of Ensemble Members:  12
Number of Ensemble Members:  3
Number of Ensemble Members:  44
Number of

In [7]:
# ── Storm selection ────────────────────────────────────────────────────────
# Available storms were printed above. Set SELECTED_STORM to one of them,
# or leave empty ("") to use the first available storm.

SELECTED_STORM = "MAILA"     # e.g. "MAILA", "VAIANU", "INDUSA", or "" for first

# Resolve storm name and filter all downstream processing to that storm only
_available_storms = sorted({
    Path(p).name.split('_storm_')[1].split('_')[0]
    for p in per_storm_csvs
    if '_storm_' in Path(p).name
})

if not _available_storms:
    raise RuntimeError(
        "No storm CSVs found — the download likely failed (404 for an old date). "
        "Update SELECTED_DATE in the Setup cell to a date within the last ~30 days "
        "when an active named storm was present."
    )

_storm = SELECTED_STORM.strip().upper() if SELECTED_STORM.strip() else _available_storms[0]
if _storm not in _available_storms:
    print(f"WARNING: '{_storm}' not found in {_available_storms}, defaulting to '{_available_storms[0]}'")
    _storm = _available_storms[0]

# Filter per_storm_csvs so Steps 3–5 only run for the selected storm
per_storm_csvs = [p for p in per_storm_csvs if f'_storm_{_storm}_' in Path(p).name]

# Dedicated subdirectories for the selected storm (keeps Step 5 isolated)
_storm_tc_dir      = Path(directories['tc_transformed']) / _storm
_storm_env_dir     = Path(directories['wind_extracted'])  / _storm
_storm_tc_dir.mkdir(exist_ok=True)
_storm_env_dir.mkdir(exist_ok=True)

print(f"Selected storm : {_storm}")
print(f"CSV to process : {[Path(p).name for p in per_storm_csvs]}")
print(f"Transform dir  : {_storm_tc_dir}")
print(f"Envelope dir   : {_storm_env_dir}")

RuntimeError: No storm CSVs found — the download likely failed (404 for an old date). Update SELECTED_DATE in the Setup cell to a date within the last ~30 days when an active named storm was present.

## Step 3: Transform Data


In [ ]:
# Transform the selected storm's CSV into Snowflake-ready format
# (standardised units, wind radii, WKT polygons)
transformation_results = []

for csv_path in per_storm_csvs:
    csv_file = Path(csv_path)
    result = transform_tc_data_from_file(
        filename=str(csv_file),
        output_dir=str(_storm_tc_dir),
        verbose=True,
    )
    if result['success']:
        transformation_results.append(result)
        print(f"  {result['records']} records → {Path(result['csv_file']).name}")
    else:
        print(f"  Failed: {csv_file.name}")

transformed_files = [Path(r['csv_file']) for r in transformation_results]
print(f"\nTransformed files: {[f.name for f in transformed_files]}")

## Step 4: Download Wind Data


In [ ]:
# Download wind forecast data for the same run as the TC data.
# forecast_hours controls how many lead times to fetch (every 6h, 0→144h = full range).
# Reduce the list here to speed up the notebook demo, e.g. list(range(0, 25, 6)).
forecast_hours = list(range(0, 145, 6))  # full range: 0, 6, 12, … 144h

if SELECTED_DATE:
    wind_date     = f"{SELECTED_DATE[:4]}-{SELECTED_DATE[4:6]}-{SELECTED_DATE[6:]}"
    wind_run_time = int(SELECTED_RUNTIME)
else:
    # If using latest TC data, derive date/run from the downloaded BUFR filename
    import re
    bufr_name  = bufr_files[0].stem          # e.g. "tc_tracks_2026-04-06_r06"
    wind_date  = re.search(r'(\d{4}-\d{2}-\d{2})', bufr_name).group(1)
    wind_run_time = int(re.search(r'_r(\d{2})', bufr_name).group(1))
    print(f"Derived from filename: date={wind_date}  run={wind_run_time:02d}Z")

wind_files = download_ensemble_wind(
    date=wind_date,
    run_time=wind_run_time,
    forecast_hours=forecast_hours,
    output_dir=directories['wind_data'],
)

## Step 5: Process Wind Combination


In [ ]:
# Create wind threshold envelopes for the selected storm only
combination_result = process_wind_combination(
    tc_data_dir=_storm_tc_dir,
    wind_data_dir=Path(directories['wind_data']),
    output_dir=_storm_env_dir,
)

## Results

In [ ]:
from visualization import show_tracks, show_tracks_with_polygons, show_individual_envelopes, show_combined_envelopes

envelope_files = list(_storm_env_dir.glob("*.csv"))

print(f"Storm: {_storm}")
print(f"TC Track Data:  {len(transformed_files)} file(s)")
print(f"Wind Envelopes: {len(envelope_files)} file(s)")
for f in transformed_files:
    print(f"  - {f.name}")
for f in envelope_files:
    print(f"  - {f.name}")

# Pick the correct files for this storm
storm_transformed_file = transformed_files[0] if transformed_files else None
storm_individual_file  = next((f for f in envelope_files if 'individual' in f.name), None)
storm_combined_file    = next((f for f in envelope_files if 'combined'   in f.name), None)

print(f"\nTracks file:     {storm_transformed_file.name if storm_transformed_file else 'NOT FOUND'}")
print(f"Individual file: {storm_individual_file.name  if storm_individual_file  else 'NOT FOUND'}")
print(f"Combined file:   {storm_combined_file.name    if storm_combined_file    else 'NOT FOUND'}")

In [ ]:
# TC track visualization for the selected storm
show_tracks(str(storm_transformed_file))

In [ ]:
# TC tracks with wind field polygons for the selected storm
show_tracks_with_polygons(str(storm_transformed_file))

In [ ]:
# Individual wind envelopes for the selected storm
show_individual_envelopes(str(storm_individual_file))

In [ ]:
# Combined wind envelopes for the selected storm
show_combined_envelopes(str(storm_combined_file))